# Week 7 final selection

Eight inputs agreed on **10 November 2026**, using the 223 official observations through round 6.
Functions 1 and 2 were agreed on 5 November. This review records forecasts at the exact prepared inputs before portal submission.

[Decision notes](README.md) · [Portal input file](submissions.txt) · [Structured record](proposals.json)


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

selection_dir = Path.cwd().resolve()
root = next(path for path in selection_dir.parents if (path / "run-python.ps1").is_file())
expected = root / ".venv/Scripts/python.exe"
if Path(sys.executable).resolve() != expected.resolve():
    raise RuntimeError(f"Use the Capstone-local interpreter: {expected}")
sys.path.insert(0, str(root / "stage-2-bbo/scripts"))
from review_week_07_queries import build_submission

record = json.loads((selection_dir / "proposals.json").read_text(encoding="utf-8"))
submission, checks = build_submission(record)
assert (selection_dir / "submissions.txt").read_bytes() == submission
print("Eight inputs verified against recorded sources, known observations and six earlier submissions.")


Eight inputs verified against recorded sources, known observations and six earlier submissions.


## Returns and forecasts

The latest return and best observed value use the original signed output. Function 1's GP and neural forecasts use the signed-log target h. Its source fit supplies a separate conditional height in raw output units. GP latent SD and neural seed SD describe different quantities.


In [2]:
rows = []
for choice in record["candidates"]:
    evidence, forecast = choice["evidence"], choice["predictions"]
    rows.append({
        "Function": choice["function"], "Agreed input": choice["portal_input"],
        "Round 6 return": evidence["latest_return"], "Rank": evidence["latest_rank"],
        "Official best": evidence["best_official_raw_output"], "Forecast units": choice["prediction_units"],
        "GP mean": forecast["gp_mean"], "GP latent SD": forecast["gp_latent_sd"],
        "Neural mean": forecast["neural_mean"], "Neural seed SD": forecast["neural_seed_sd"],
    })
numbers = ["Round 6 return", "Official best", "GP mean", "GP latent SD", "Neural mean", "Neural seed SD"]
display(pd.DataFrame(rows).style.format({column: "{:.7g}" for column in numbers}))
print(f'Function 1 conditional positive source height: {record["candidates"][0]["predictions"]["positive_source_height"]:.7g}')


,Function,Agreed input,Round 6 return,Rank,Official best,Forecast units,GP mean,GP latent SD,Neural mean,Neural seed SD
0,1,0.627916-0.628953,2.170913e-14,1,2.170913e-14,signed-log height h,-218.1837,100.507,21.39579,37.78464
1,2,0.716870-0.000000,0.7230575,1,0.7230575,output y,0.7034546,0.03776482,0.6675122,0.03021045
2,3,0.495233-0.568579-0.295191,-0.1214168,19,-0.03483531,output y,-0.03288702,0.03073071,-0.05490418,0.006314155
3,4,0.419096-0.406060-0.364200-0.426466,0.4899061,2,0.6501304,output y,-0.2842222,0.3642125,0.2709121,0.3810236
4,5,0.999999-0.999999-0.979593-0.999999,1616.626,6,8662.405,output y,8368.779,159.8533,8351.005,4.229319
5,6,0.414821-0.260324-0.607614-0.728412-0.000000,-0.3281924,1,-0.3281924,output y,-0.3639076,0.04749846,-0.3412956,0.007582843
6,7,0.000000-0.135660-0.364055-0.203076-0.342712-0.663441,2.739375,1,2.739375,output y,2.64269,0.1467743,2.776517,0.01409707
7,8,0.000000-0.126300-0.212455-0.132451-0.899004-0.732961-0.231231-0.808273,9.782943,1,9.782943,output y,9.878856,0.1142186,9.792058,0.006424717


Function 1 conditional positive source height: 0.7080881


## Selection evidence and checks

Each choice links its saved model evidence and records its caveats. The Function 8 screen is reproduced from its existing candidate pool. The remaining allowance is seven queries per function including this batch.


In [3]:
display(pd.DataFrame([{
    "Function": choice["function"], "Policy": choice["selection_policy"],
    "Reason": choice["reason"], "Caveats": " ".join(choice["caveats"]),
} for choice in record["candidates"]]))
display(pd.DataFrame(checks))
display(pd.Series(record["candidates"][7]["evidence"]["local_screen"], name="Function 8 local screen"))


,Function,Policy,Reason,Caveats
0,1,test_official_only_positive_source_hypothesis,Test the analytic peak of a circular Gaussian ...,The positive component cannot explain negative...
1,2,retained_gp_ucb,Refine x1 near the successful lower-x2 query u...,The fitted mean is below the measured incumben...
2,3,near_incumbent_prior_mean_candidate,Revisit the previous mean-led option near the ...,The refreshed GP mean offers a small gain over...
3,4,incumbent_x3_test_informed_by_retained_peer,"Hold x1, x2 and x4 at the official incumbent a...",The peer result changed other coordinates as w...
4,5,retained_gp_ucb_isolated_x3_test,Complete the postponed isolated x3 reduction a...,Both saved fits predict a lower score than the...
5,6,local_neural_candidate_with_gp_crosscheck,Follow the successful neural-led trial with th...,Both fitted means are below the new incumbent....
6,7,retained_gp_ucb,Retain the local GP-UCB proposal at weight 1.9...,The GP mean is below the measured best and the...
7,8,local_gp_mean_with_neural_screen,Use the highest GP mean in the existing local ...,Both predicted improvements are small relative...


,function,known_rows_checked,prior_submission_files_checked,duplicate,nearest_known_input_distance
0,1,22,6,False,0.052451
1,2,21,6,False,0.016350
2,3,26,6,False,0.062297
3,4,42,6,False,0.017799
4,5,26,6,False,0.020406
5,6,26,6,False,0.048407
6,7,36,6,False,0.108980
7,8,46,6,False,0.239956


pool_rows                     20986.00
local_radius                      0.15
local_rows                     4352.00
both_means_above_incumbent       67.00
Name: Function 8 local screen, dtype: float64

## Course lifecycle

1. **Define the goal and scope** — agree one bounded input per function for Week 7.
2. **Gather the data** — reconcile the round 6 results, official snapshots, retained peer rows and saved fits.
3. **Explore the data** — compare the latest ranks, incumbents and local alternatives.
4. **Clean and preprocess the data** — retain exact observation values and check the pinned snapshot hashes.
5. **Select and engineer features** — reuse the fitted input transformations and the Function 1 signed-log target.
6. **Define the machine-learning task** — use regression and measured evidence to guide signed-output maximisation.
7. **Partition the data** — retain the original leave-one-out evidence. A new split is not applicable to this selection review.
8. **Select and train candidate methods** — select among saved-fit forecasts and controlled experiments. New training is not applicable.
9. **Evaluate and interpret the results** — cross-check the choices, forecast units, uncertainty, spacing and exact inputs.
10. **Deploy and iterate** — prepare the eight portal lines and retain the forecasts for the next official return.
